# Circulai φ — analyse de puissance (A/B, deux proportions)

Miroir de `scripts/circulai_power_analysis.py` : **statsmodels** `NormalIndPower` et `proportion_effectsize`.

Installation : `pip install statsmodels numpy` (scipy souvent installé en dépendance ; optionnel explicitement).

Runbook canary : [`playbook-canary.md`](playbook-canary.md).

In [ ]:
import math

import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

p_control = 0.12
min_lift_relative = 0.15
alpha = 0.05
power = 0.80
ratio = 1.0
daily_visitors = 5000
pct_eligible = 0.25
alternative = "larger"  # "larger" | "smaller" | "two-sided"

p_phi = float(np.clip(p_control * (1.0 + min_lift_relative), 0.0, 1.0))
# Convention Cohen h statsmodels : arcsin(√p1) − arcsin(√p2) ; (p_phi, p_control) > 0 si p_phi > p_control.
effect_size = proportion_effectsize(p_phi, p_control)

analysis = NormalIndPower()
n_raw = analysis.solve_power(
    effect_size=effect_size,
    power=power,
    alpha=alpha,
    ratio=ratio,
    alternative=alternative,
)
n_control = float(np.squeeze(np.asarray(n_raw)))
n_treatment = n_control * ratio
n_total = n_control + n_treatment
eligible_per_day = daily_visitors * pct_eligible
days_hint = math.ceil(n_total / eligible_per_day) if eligible_per_day > 0 else math.nan

print("p_control, p_phi:", p_control, p_phi)
print("effect_size:", effect_size)
print("n control / treatment / total:", math.ceil(n_control), math.ceil(n_treatment), math.ceil(n_total))
print("duration hint (days):", days_hint)

## Extensions (optionnel)

- Ajuster `ratio` si les effectifs ne sont pas symétriques entre les bras.
- `alternative="two-sided"` pour une hypothèse bilatérale (souvent effectif plus élevé).
- Sensibilité : faire varier `p_control` et `min_lift_relative` ; tracer une courbe puissance avec `matplotlib` si besoin pédagogique.